# 🗄️ Qdrant Vector Indexing & Semantic Search Pipeline

This notebook demonstrates how to load the enriched chunks from `processed_documents.json`, embed them using **`nvidia/llama-nemotron-embed-1b-v2`**, and index them into a **local Qdrant Vector Database**.

In [1]:
pip install -q qdrant-client langchain-qdrant langchain-nvidia-ai-endpoints

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2

import os
import importlib
from dotenv import load_dotenv
import qdrant_indexer
importlib.reload(qdrant_indexer)
from qdrant_indexer import QdrantIndexer

load_dotenv()

# 1. Initialize Qdrant Indexer (Defaults to Local Storage at Data/qdrant_db)
indexer = QdrantIndexer(
    collection_name="fintech_documents",
    embedding_model="nvidia/llama-nemotron-embed-1b-v2",
    qdrant_path="Data/qdrant_db",
    batch_size=100
)

Initializing NVIDIAEmbeddings with model: nvidia/llama-nemotron-embed-1b-v2
Initializing local Qdrant Client at: Data/qdrant_db


In [3]:
# 2. Load enriched JSON chunks into Document objects
docs = indexer.load_documents("Data/processed_documents.json")

print("\n--- Sample Document Metadata ---")
print(docs[0].metadata)


[1/3] Loading enriched documents from 'Data/processed_documents.json'...
Loaded 2258 documents.

--- Sample Document Metadata ---
{'source': 'Data/Test/RBI-GUIDELINES-ON-DIGITAL-LENDING-02-09-22.pdf', 'filename': 'RBI-GUIDELINES-ON-DIGITAL-LENDING-02-09-22.pdf', 'page': 1, 'total_pages': 12, 'file_type': 'pdf', 'chunk_index': 1, 'document_title': 'Guidelines on Digital Lending', 'document_type': 'Regulatory Guideline', 'issuing_authority': 'Reserve Bank of India (RBI)', 'primary_domain': 'Digital Lending', 'doc_executive_summary': 'The RBI has issued guidelines on digital lending to ensure compliance with extant guidelines on outsourcing and to ensure a smooth transition for regulated entities. The guidelines are applicable to existing and new customers and require regulated entities to put in place adequate systems and processes to ensure compliance.', 'chunk_title': 'RBI Guidelines on Digital Lending', 'chunk_summary': 'RBI circular on digital lending guidelines for commercial banks

In [4]:
# 3. Embed & Index into Qdrant
indexer.index_documents(docs)


[2/3] Indexing 2258 documents into Qdrant (fintech_documents)...
  -> Inferring embedding dimensions...
  -> Indexing batch 1/23 (chunks 0 to 99)...
  -> Indexing batch 2/23 (chunks 100 to 199)...
  -> Indexing batch 3/23 (chunks 200 to 299)...
  -> Indexing batch 4/23 (chunks 300 to 399)...
  -> Indexing batch 5/23 (chunks 400 to 499)...
  -> Indexing batch 6/23 (chunks 500 to 599)...
  -> Indexing batch 7/23 (chunks 600 to 699)...
  -> Indexing batch 8/23 (chunks 700 to 799)...
  -> Indexing batch 9/23 (chunks 800 to 899)...
  -> Indexing batch 10/23 (chunks 900 to 999)...
  -> Indexing batch 11/23 (chunks 1000 to 1099)...
  -> Indexing batch 12/23 (chunks 1100 to 1199)...
  -> Indexing batch 13/23 (chunks 1200 to 1299)...
  -> Indexing batch 14/23 (chunks 1300 to 1399)...
  -> Indexing batch 15/23 (chunks 1400 to 1499)...
  -> Indexing batch 16/23 (chunks 1500 to 1599)...
  -> Indexing batch 17/23 (chunks 1600 to 1699)...
  -> Indexing batch 18/23 (chunks 1700 to 1799)...
  -> Inde

In [6]:
# 4. Perform a Semantic Search against Qdrant
query = "What is my Name ?"
results = indexer.search(query, k=3)

for i, res in enumerate(results, 1):
    print(f"\n--- Result {i}: {res.metadata.get('chunk_title')} ---")
    print(f"Domain: {res.metadata.get('primary_domain')}")
    print(f"Entities: {res.metadata.get('entities')}")
    print(f"Mandates: {res.metadata.get('compliance_mandates')}")
    print(f"Content Excerpt: {res.page_content[:200]}...")


[Search] 'What is my Name ?'
Search completed in 0.63 seconds.

--- Result 1: HDFC Bank IR26 - Director Changes ---
Domain: Digital Lending, Fintech, Banking, Corporate Finance
Entities: ['HDFC Bank', 'Mythra Mahesh', 'Mr. Mahesh Babu', 'Ramamurthy', 'Nagsri', 'C Jagadisan HUF', 'Ms. Havovi Bharucha']
Mandates: ['CEOs and Directors', 'Corporate Governance', 'Financial Reporting']
Content Excerpt: Mythra Mahesh, Mr. Mahesh Babu 
Ramamurthy (ceased with effect from June 30, 2025), Nagsri - Creating Special Memories, C Jagadisan HUF (with effect from 
December 29, 2025), Ms. Havovi Bharucha, Mr....

--- Result 2: Directors of HDFC Bank IR26 ---
Domain: Digital Lending, Fintech, Banking, Corporate Finance
Entities: ['HDFC Bank IR26', 'Mythra Mahesh', 'Mr. Mahesh Babu', 'Ramamurthy', 'Nagsri', 'C Jagadisan HUF', 'Ms. Havovi Bharucha']
Mandates: ['Corporate Governance Requirements', 'Director Tenure', 'Director Appointment']
Content Excerpt: Mythra Mahesh, Mr. Mahesh Babu 
Ramamurthy (cease